In [2]:
println!("Hello from Rust in Cursor Jupyter!");

Hello from Rust in Cursor Jupyter!


In [ ]:
use std::io::Write;
use std::process::{Command, Stdio};
use std::time::{SystemTime, UNIX_EPOCH};

/// 测试 Binance API Key/Secret 是否可用（不会打印密钥）。
///
/// 调用规则：
/// 1) query string 包含 timestamp(毫秒) 和 recvWindow
/// 2) signature = HMAC_SHA256(secret, query_string)
/// 3) Header 需要 X-MBX-APIKEY
fn test_binance_api_credentials(api_key: &str, secret_key: &str) -> Result<String, String> {
    if api_key.trim().is_empty() || secret_key.trim().is_empty() {
        return Err("API Key 或 Secret 为空".to_string());
    }

    let ts_ms = SystemTime::now()
        .duration_since(UNIX_EPOCH)
        .map_err(|e| format!("读取系统时间失败: {e}"))?
        .as_millis();

    let query = format!("timestamp={ts_ms}&recvWindow=5000");
    let signature = hmac_sha256_hex_with_openssl(secret_key, &query)?;

    let url = format!(
        "https://api.binance.com/api/v3/account?{}&signature={}",
        query, signature
    );

    // 用 curl 发请求，并在最后一行附加 HTTP 状态码。
    let output = Command::new("curl")
        .args([
            "-sS",
            "-H",
            &format!("X-MBX-APIKEY: {api_key}"),
            "-w",
            "\n%{http_code}",
            &url,
        ])
        .output()
        .map_err(|e| format!("调用 curl 失败: {e}"))?;

    if !output.status.success() {
        let stderr = String::from_utf8_lossy(&output.stderr);
        return Err(format!("curl 进程失败: {stderr}"));
    }

    let raw = String::from_utf8_lossy(&output.stdout).to_string();
    let mut lines: Vec<&str> = raw.lines().collect();
    if lines.is_empty() {
        return Err("空响应".to_string());
    }

    let status = lines.pop().unwrap_or("000").trim().to_string();
    let body = lines.join("\n");

    if status == "200" {
        Ok(format!("Binance 鉴权成功，HTTP {status}\n响应摘要: {}", summarize_body(&body)))
    } else {
        Err(format!("Binance 鉴权失败，HTTP {status}\n响应: {body}"))
    }
}

fn hmac_sha256_hex_with_openssl(secret_key: &str, message: &str) -> Result<String, String> {
    let mut child = Command::new("openssl")
        .args(["dgst", "-sha256", "-hmac", secret_key])
        .stdin(Stdio::piped())
        .stdout(Stdio::piped())
        .stderr(Stdio::piped())
        .spawn()
        .map_err(|e| format!("启动 openssl 失败: {e}"))?;

    if let Some(stdin) = child.stdin.as_mut() {
        stdin
            .write_all(message.as_bytes())
            .map_err(|e| format!("写入 openssl stdin 失败: {e}"))?;
    }

    let out = child
        .wait_with_output()
        .map_err(|e| format!("等待 openssl 输出失败: {e}"))?;

    if !out.status.success() {
        let stderr = String::from_utf8_lossy(&out.stderr);
        return Err(format!("openssl 失败: {stderr}"));
    }

    // openssl 输出通常是："SHA2-256(stdin)= <hex>"
    let text = String::from_utf8_lossy(&out.stdout);
    let hex = text
        .split('=' )
        .nth(1)
        .map(str::trim)
        .ok_or_else(|| format!("无法解析 openssl 输出: {text}"))?;

    Ok(hex.to_string())
}

fn summarize_body(body: &str) -> String {
    // 避免输出过长，保留前 200 字符用于 notebook 展示
    let preview: String = body.chars().take(200).collect();
    if body.chars().count() > 200 {
        format!("{preview}...")
    } else {
        preview
    }
}

// ===== 调用示例（请填你自己的 key/secret） =====
let api_key = "";
let secret_key = "";

match test_binance_api_credentials(api_key, secret_key) {
    Ok(msg) => println!("✅ {msg}"),
    Err(err) => println!("❌ {err}"),
}

✅ Binance 鉴权成功，HTTP 200
响应摘要: {"makerCommission":10,"takerCommission":10,"buyerCommission":0,"sellerCommission":0,"commissionRates":{"maker":"0.00100000","taker":"0.00100000","buyer":"0.00000000","seller":"0.00000000"},"canTrade":...


()

In [12]:
:dep rusqlite = "0.31"
:dep aes-gcm = "0.10"
:dep base64 = "0.22"
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"

use aes_gcm::aead::{Aead, KeyInit};
use aes_gcm::{Aes256Gcm, Key, Nonce};
use base64::engine::general_purpose::STANDARD as B64;
use base64::Engine;
use rusqlite::{params, Connection};
use serde::Deserialize;
use std::collections::HashMap;

#[derive(Debug, Deserialize)]
struct EncryptedCredentialPayload {
    v: u8,
    alg: String,
    nonce_b64: String,
    ciphertext_b64: String,
    updated_at: i64,
}

#[derive(Debug, Deserialize)]
struct PlainCredentialPayload {
    envs: HashMap<String, String>,
}

/// 从 app_settings 读取指定 key。
fn get_setting(conn: &Connection, key: &str) -> Result<String, String> {
    conn.query_row(
        "SELECT value FROM app_settings WHERE key = ?1",
        params![key],
        |r| r.get::<_, String>(0),
    )
    .map_err(|e| format!("读取设置 {key} 失败: {e}"))
}

/// 将 base64 主密钥解码为 32 字节。
fn decode_master_key_b64(raw: &str) -> Result<[u8; 32], String> {
    let decoded = B64
        .decode(raw.trim())
        .map_err(|e| format!("master key base64 解码失败: {e}"))?;
    if decoded.len() != 32 {
        return Err(format!(
            "master key 长度错误，期望 32 字节，实际 {} 字节",
            decoded.len()
        ));
    }
    let mut out = [0_u8; 32];
    out.copy_from_slice(&decoded);
    Ok(out)
}

/// 解密数据库里的 cred:* 密文。
fn decrypt_credential_payload(
    encrypted_json: &str,
    master_key_b64: &str,
) -> Result<HashMap<String, String>, String> {
    let payload: EncryptedCredentialPayload =
        serde_json::from_str(encrypted_json).map_err(|e| format!("解析密文 JSON 失败: {e}"))?;

    if payload.v != 1 || payload.alg != "AES-256-GCM" {
        return Err(format!(
            "不支持的密文版本/算法: v={}, alg={}",
            payload.v, payload.alg
        ));
    }

    let nonce_bytes = B64
        .decode(payload.nonce_b64.trim())
        .map_err(|e| format!("nonce base64 解码失败: {e}"))?;
    if nonce_bytes.len() != 12 {
        return Err(format!("nonce 长度错误，期望 12，实际 {}", nonce_bytes.len()));
    }

    let ciphertext = B64
        .decode(payload.ciphertext_b64.trim())
        .map_err(|e| format!("ciphertext base64 解码失败: {e}"))?;

    let key_bytes = decode_master_key_b64(master_key_b64)?;
    let key = Key::<Aes256Gcm>::from_slice(&key_bytes);
    let cipher = Aes256Gcm::new(key);
    let plain = cipher
        .decrypt(Nonce::from_slice(&nonce_bytes), ciphertext.as_ref())
        .map_err(|_| "AES-GCM 解密失败".to_string())?;

    let parsed: PlainCredentialPayload =
        serde_json::from_slice(&plain).map_err(|e| format!("解析明文 JSON 失败: {e}"))?;
    Ok(parsed.envs)
}

/// 在 env map 中按多个候选 key 取值。
fn pick_first<'a>(envs: &'a HashMap<String, String>, keys: &[&str]) -> Option<&'a String> {
    keys.iter().find_map(|k| envs.get(*k))
}

/// 解析 SQLite 路径：优先 `DATABASE_PATH`，否则从当前目录向上查找 `data/server.db`。
fn resolve_server_db_path() -> Result<std::path::PathBuf, String> {
    use std::path::PathBuf;

    if let Ok(p) = std::env::var("DATABASE_PATH") {
        let p = PathBuf::from(p.trim());
        if p.is_file() {
            return Ok(p);
        }
        return Err(format!("DATABASE_PATH 不存在或不是文件: {}", p.display()));
    }

    let cwd = std::env::current_dir().map_err(|e| format!("获取当前目录失败: {e}"))?;
    let mut dir = cwd.clone();
    loop {
        let cand = dir.join("data/server.db");
        if cand.is_file() {
            return Ok(cand);
        }
        if !dir.pop() {
            break;
        }
    }
    Err(format!(
        "找不到 data/server.db（已从 {} 向上搜索父目录）。请在仓库根目录启动 kernel，或设置环境变量 DATABASE_PATH",
        cwd.display()
    ))
}

let exchange = "binance"; // 可改成 okx / polyclaw

let db_path = resolve_server_db_path()?;
let conn = Connection::open(&db_path).map_err(|e| {
    format!(
        "打开数据库失败: {e} (path={}, cwd={})",
        db_path.display(),
        std::env::current_dir()
            .map(|p| p.display().to_string())
            .unwrap_or_else(|_| "?".into())
    )
})?;
let master_key_b64 = get_setting(&conn, "cred:master_key_b64")?;
let encrypted = get_setting(&conn, &format!("cred:{exchange}"))?;
let envs = decrypt_credential_payload(&encrypted, &master_key_b64)?;

let api_key = pick_first(
    &envs,
    &[
        "BINANCE_API_KEY",
        "API_KEY",
        "api_key",
        "binance_api_key",
    ],
)
.cloned()
.unwrap_or_default();
let secret_key = pick_first(
    &envs,
    &[
        "BINANCE_SECRET_KEY",
        "SECRET_KEY",
        "secret_key",
        "binance_secret_key",
    ],
)
.cloned()
.unwrap_or_default();

println!("exchange={exchange}");
println!("api_key={api_key}");
println!("secret_key={secret_key}");


exchange=binance
api_key=Q29c2kEDBKKwWz5Z7pWRm3DWFkND6Tke6AmVGfI4dzocBmQhIZktcNVVHjNasRYr
secret_key=0nMb324phQFQcW732EtFN5ha2MZhuE6DnZYyNHCwtubSz5EMRlW2guuRPWOVKUma


In [14]:
:dep rusqlite = "0.31"
:dep aes-gcm = "0.10"
:dep base64 = "0.22"
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"
:dep hmac = "0.12"
:dep sha2 = "0.10"
:dep chrono = { version = "0.4", default-features = false, features = ["clock"] }
:dep reqwest = { version = "0.12", default-features = false, features = ["blocking", "rustls-tls"] }

use aes_gcm::aead::{Aead, KeyInit};
use aes_gcm::{Aes256Gcm, Key, Nonce};
use base64::engine::general_purpose::STANDARD as B64;
use base64::Engine;
use chrono::{Local, LocalResult, SecondsFormat, TimeZone};
use hmac::{Hmac, Mac};
use rusqlite::{params, Connection};
use serde::Deserialize;
use serde_json::Value;
use sha2::Sha256;
use std::collections::HashMap;
use std::time::Duration;

type HmacSha256 = Hmac<Sha256>;

#[derive(Debug, Deserialize)]
struct EncryptedCredentialPayload {
    v: u8,
    alg: String,
    nonce_b64: String,
    ciphertext_b64: String,
}

#[derive(Debug, Deserialize)]
struct PlainCredentialPayload {
    envs: HashMap<String, String>,
}

fn get_setting(conn: &Connection, key: &str) -> Result<String, String> {
    conn.query_row(
        "SELECT value FROM app_settings WHERE key = ?1",
        params![key],
        |r| r.get::<_, String>(0),
    )
    .map_err(|e| format!("读取设置 {key} 失败: {e}"))
}

fn decode_master_key_b64(raw: &str) -> Result<[u8; 32], String> {
    let decoded = B64
        .decode(raw.trim())
        .map_err(|e| format!("master key base64 解码失败: {e}"))?;
    if decoded.len() != 32 {
        return Err(format!("master key 长度错误，期望 32 字节，实际 {} 字节", decoded.len()));
    }
    let mut out = [0_u8; 32];
    out.copy_from_slice(&decoded);
    Ok(out)
}

fn decrypt_credential_payload(
    encrypted_json: &str,
    master_key_b64: &str,
) -> Result<HashMap<String, String>, String> {
    let payload: EncryptedCredentialPayload =
        serde_json::from_str(encrypted_json).map_err(|e| format!("解析密文 JSON 失败: {e}"))?;

    if payload.v != 1 || payload.alg != "AES-256-GCM" {
        return Err(format!("不支持的密文版本/算法: v={}, alg={}", payload.v, payload.alg));
    }

    let nonce_bytes = B64
        .decode(payload.nonce_b64.trim())
        .map_err(|e| format!("nonce base64 解码失败: {e}"))?;
    let ciphertext = B64
        .decode(payload.ciphertext_b64.trim())
        .map_err(|e| format!("ciphertext base64 解码失败: {e}"))?;

    let key_bytes = decode_master_key_b64(master_key_b64)?;
    let key = Key::<Aes256Gcm>::from_slice(&key_bytes);
    let cipher = Aes256Gcm::new(key);
    let plain = cipher
        .decrypt(Nonce::from_slice(&nonce_bytes), ciphertext.as_ref())
        .map_err(|_| "AES-GCM 解密失败".to_string())?;

    let parsed: PlainCredentialPayload =
        serde_json::from_slice(&plain).map_err(|e| format!("解析明文 JSON 失败: {e}"))?;
    Ok(parsed.envs)
}

fn pick_first<'a>(envs: &'a HashMap<String, String>, keys: &[&str]) -> Option<&'a String> {
    keys.iter().find_map(|k| envs.get(*k))
}

/// 解析 SQLite 路径：优先 `DATABASE_PATH`，否则从当前目录向上查找 `data/server.db`。
fn resolve_server_db_path() -> Result<std::path::PathBuf, String> {
    use std::path::PathBuf;

    if let Ok(p) = std::env::var("DATABASE_PATH") {
        let p = PathBuf::from(p.trim());
        if p.is_file() {
            return Ok(p);
        }
        return Err(format!("DATABASE_PATH 不存在或不是文件: {}", p.display()));
    }

    let cwd = std::env::current_dir().map_err(|e| format!("获取当前目录失败: {e}"))?;
    let mut dir = cwd.clone();
    loop {
        let cand = dir.join("data/server.db");
        if cand.is_file() {
            return Ok(cand);
        }
        if !dir.pop() {
            break;
        }
    }
    Err(format!(
        "找不到 data/server.db（已从 {} 向上搜索父目录）。请在仓库根目录启动 kernel，或设置环境变量 DATABASE_PATH",
        cwd.display()
    ))
}

fn okx_sign(ts: &str, method: &str, req_path: &str, body: &str, secret: &str) -> Result<String, String> {
    let pre_hash = format!("{ts}{method}{req_path}{body}");
    let mut mac = <HmacSha256 as Mac>::new_from_slice(secret.as_bytes())
        .map_err(|_| "无效 SECRET_KEY".to_string())?;
    mac.update(pre_hash.as_bytes());
    Ok(B64.encode(mac.finalize().into_bytes()))
}

/// 已签名的 GET，返回 JSON（校验 HTTP 200 且 `code == "0"`）。`path_with_query` 须含前导 `/`，如 `/api/v5/asset/balances`。
///
/// 使用 `reqwest` + **rustls** 发 HTTPS，避免 macOS 自带 `curl`/LibreSSL 出现 `SSL_ERROR_SYSCALL`。
fn okx_get_json(
    api_key: &str,
    secret_key: &str,
    passphrase: &str,
    path_with_query: &str,
) -> Result<Value, String> {
    if api_key.trim().is_empty() || secret_key.trim().is_empty() || passphrase.trim().is_empty() {
        return Err("API_KEY / SECRET_KEY / PASSPHRASE 不能为空".to_string());
    }

    let ts = chrono::Utc::now().to_rfc3339_opts(SecondsFormat::Millis, true);
    let sign = okx_sign(&ts, "GET", path_with_query, "", secret_key)?;
    let url = format!("https://www.okx.com{path_with_query}");

    let client = reqwest::blocking::Client::builder()
        .timeout(Duration::from_secs(45))
        .connect_timeout(Duration::from_secs(20))
        .build()
        .map_err(|e| format!("HTTP 客户端初始化失败: {e}"))?;

    let resp = client
        .get(url)
        .header("OK-ACCESS-KEY", api_key)
        .header("OK-ACCESS-SIGN", sign)
        .header("OK-ACCESS-TIMESTAMP", &ts)
        .header("OK-ACCESS-PASSPHRASE", passphrase)
        .send()
        .map_err(|e| {
            format!(
                "HTTPS 请求失败: {e}（已改用 rustls；若在公司网可设置 HTTPS_PROXY，或检查防火墙/VPN）"
            )
        })?;

    let status = resp.status().as_u16();
    let body = resp
        .text()
        .map_err(|e| format!("读取响应体失败: {e}"))?;

    if status != 200 {
        return Err(format!("HTTP {status}\n{body}"));
    }

    let v: Value = serde_json::from_str(&body).map_err(|e| format!("解析 JSON 失败: {e}\n{body}"))?;
    let code = v["code"].as_str().unwrap_or("");
    if code != "0" {
        return Err(format!(
            "OKX 业务错误 code={code} msg={}",
            v["msg"].as_str().unwrap_or("")
        ));
    }
    Ok(v)
}

fn j_field(v: &Value, key: &str) -> String {
    v.get(key)
        .map(|x| match x {
            Value::String(s) => s.clone(),
            Value::Number(n) => n.to_string(),
            Value::Bool(b) => b.to_string(),
            Value::Null => String::new(),
            _ => x.to_string(),
        })
        .unwrap_or_default()
}

/// 资金账户：各代币余额（`/api/v5/asset/balances`）。
fn print_okx_funding_assets(j: &Value) {
    println!("\n=== 资金账户代币 (GET /api/v5/asset/balances) ===");
    let Some(data) = j.get("data").and_then(|d| d.as_array()) else {
        println!("(无 data)");
        return;
    };
    println!(
        "{:<10} {:>22} {:>22} {:>22}",
        "ccy", "bal", "availBal", "frozenBal"
    );
    for row in data {
        let ccy = j_field(row, "ccy");
        if ccy.is_empty() {
            continue;
        }
        let bal = j_field(row, "bal");
        let avail = j_field(row, "availBal");
        let frozen = j_field(row, "frozenBal");
        if bal == "0" && avail == "0" && frozen == "0" {
            continue;
        }
        println!("{ccy:<10} {bal:>22} {avail:>22} {frozen:>22}");
    }
}

/// 总资产估值：交易/资金/经典/赚币等账户折合为同一计价币（`/api/v5/asset/asset-valuation?ccy=`）。
fn print_okx_asset_valuation(j: &Value, valuation_ccy: &str) {
    println!(
        "\n=== 总资产估值 (GET /api/v5/asset/asset-valuation?ccy={valuation_ccy}) ==="
    );
    let Some(data) = j.get("data").and_then(|d| d.as_array()) else {
        println!("(无 data)");
        return;
    };
    for row in data {
        println!(
            "totalBal: {} {}",
            j_field(row, "totalBal"),
            valuation_ccy
        );
        let ts = j_field(row, "ts");
        if !ts.is_empty() {
            println!("ts (ms): {ts}");
        }
        let Some(det) = row.get("details") else {
            continue;
        };
        match det {
            Value::Object(map) => {
                println!("分项 ({valuation_ccy}):");
                let mut keys: Vec<&String> = map.keys().collect();
                keys.sort();
                for k in keys {
                    let v = &map[k];
                    let val = match v {
                        Value::String(s) => s.clone(),
                        Value::Number(n) => n.to_string(),
                        _ => v.to_string(),
                    };
                    println!("  {:12} {val}", k);
                }
            }
            Value::Array(arr) => {
                println!("details (原始列表):");
                for d in arr {
                    println!("  {}", d);
                }
            }
            _ => println!("details: {det}"),
        }
    }
}

/// 今日总盈亏 = **已实现(pnl+fee)** + **当前未实现(Σ upl)**。
///
/// 1) `GET /api/v5/account/bills`：**不传 `instType`**（欧易不接受 `ANY`，省略该参数即查全品类）、`type=2`、`begin`/`end` 为本地今日 0 点至今（毫秒）。
/// 2) `GET /api/v5/account/positions`：累加 `upl`，并逐行展示 `upl` / `uplRatio`。
fn print_okx_today_total_pnl(
    api_key: &str,
    secret_key: &str,
    passphrase: &str,
) -> Result<(), String> {
    let naive = Local::now()
        .date_naive()
        .and_hms_opt(0, 0, 0)
        .ok_or_else(|| "今日 0 点时间无效".to_string())?;
    let begin_dt = match Local.from_local_datetime(&naive) {
        LocalResult::Single(dt) => dt,
        LocalResult::Ambiguous(earliest, _) => earliest,
        LocalResult::None => return Err("无法解析今日 0 点（本地时区）".to_string()),
    };
    let begin_ms = begin_dt.timestamp_millis();
    let end_ms = Local::now().timestamp_millis();

    println!("\n=== 今日总盈亏（已实现含手续费 + 持仓未实现）===");
    println!(
        "本地自然日: {} 00:00 — 现在 (begin={begin_ms} end={end_ms} ms)",
        begin_dt.format("%Y-%m-%d")
    );

    // —— 1) 今日已实现：bills type=2，全品类 ANY ——
    println!("\n--- 1) 今日已实现 (GET /api/v5/account/bills, 无 instType=全品类, type=2) ---");
    let mut sum_pnl = 0_f64;
    let mut sum_fee = 0_f64;
    let mut bill_rows = 0_usize;
    let mut after_cursor: Option<String> = None;
    let max_pages = 50_usize;
    let mut page = 0_usize;

    loop {
        page += 1;
        if page > max_pages {
            println!("(bills 已达分页上限 {max_pages} 页，可增大 max_pages)");
            break;
        }
        let mut path =
            format!("/api/v5/account/bills?begin={begin_ms}&end={end_ms}&type=2&limit=100");
        if let Some(ref a) = after_cursor {
            path.push_str(&format!("&after={a}"));
        }
        let j = okx_get_json(api_key, secret_key, passphrase, &path)?;
        let Some(data) = j.get("data").and_then(|d| d.as_array()) else {
            break;
        };
        if data.is_empty() {
            break;
        }
        for row in data {
            sum_pnl += j_field(row, "pnl").parse::<f64>().unwrap_or(0.0);
            sum_fee += j_field(row, "fee").parse::<f64>().unwrap_or(0.0);
            bill_rows += 1;
        }
        if data.len() < 100 {
            break;
        }
        let oldest = j_field(data.last().unwrap(), "billId");
        if oldest.is_empty() {
            break;
        }
        after_cursor = Some(oldest);
    }

    let realized_incl_fee = sum_pnl + sum_fee;
    println!("账单条数: {bill_rows}");
    println!("Σ pnl: {sum_pnl}");
    println!("Σ fee: {sum_fee}");
    println!("今日已实现(含手续费, pnl+fee): {realized_incl_fee}");

    // —— 2) 当前未实现：positions ——
    println!("\n--- 2) 当前未实现 (GET /api/v5/account/positions) ---");
    let pos_j = okx_get_json(api_key, secret_key, passphrase, "/api/v5/account/positions")?;
    let mut sum_upl = 0_f64;
    if let Some(data) = pos_j.get("data").and_then(|d| d.as_array()) {
        println!("{:<28} {:>14} {:>14}", "instId", "upl", "uplRatio");
        for row in data {
            let upl = j_field(row, "upl").parse::<f64>().unwrap_or(0.0);
            sum_upl += upl;
            let inst = j_field(row, "instId");
            let ratio = j_field(row, "uplRatio");
            if !inst.is_empty() {
                println!("{inst:<28} {upl:>14} {ratio:>14}");
            }
        }
    } else {
        println!("(无 positions data)");
    }
    println!("Σ upl (未实现合计): {sum_upl}");

    let today_total = realized_incl_fee + sum_upl;
    println!("\n>>> 今日总盈亏 (已实现含费 + 未实现): {today_total}");
    Ok(())
}

let db_path = resolve_server_db_path()?;
let conn = Connection::open(&db_path).map_err(|e| {
    format!(
        "打开数据库失败: {e} (path={}, cwd={})",
        db_path.display(),
        std::env::current_dir()
            .map(|p| p.display().to_string())
            .unwrap_or_else(|_| "?".into())
    )
})?;
let master_key_b64 = get_setting(&conn, "cred:master_key_b64")?;
let encrypted = get_setting(&conn, "cred:okx")?;
let envs = decrypt_credential_payload(&encrypted, &master_key_b64)?;

let api_key = pick_first(&envs, &["API_KEY", "api_key"]).cloned().unwrap_or_default();
let secret_key = pick_first(&envs, &["SECRET_KEY", "secret_key"]).cloned().unwrap_or_default();
let passphrase = pick_first(&envs, &["PASSPHRASE", "passphrase", "phrase"]).cloned().unwrap_or_default();

println!(
    "api_key.len={} secret.len={} passphrase.len={}",
    api_key.len(),
    secret_key.len(),
    passphrase.len()
);

// 计价币种：总资产折合为该币种数量（如 USDT、BTC、ETH）
let valuation_ccy = "USDT";
match okx_get_json(
    &api_key,
    &secret_key,
    &passphrase,
    &format!("/api/v5/asset/asset-valuation?ccy={valuation_ccy}"),
) {
    Ok(j) => print_okx_asset_valuation(&j, valuation_ccy),
    Err(e) => println!("❌ 总资产估值: {e}"),
}

if let Err(e) = print_okx_today_total_pnl(&api_key, &secret_key, &passphrase) {
    println!("❌ 今日总盈亏: {e}");
}

match okx_get_json(&api_key, &secret_key, &passphrase, "/api/v5/asset/balances") {
    Ok(j) => print_okx_funding_assets(&j),
    Err(e) => println!("❌ 资金账户代币: {e}"),
}

api_key.len=36 secret.len=32 passphrase.len=15

=== 总资产估值 (GET /api/v5/asset/asset-valuation?ccy=USDT) ===
totalBal: 3729.16000000 USDT
ts (ms): 1778132372452
分项 (USDT):
  classic      0
  earn         3727
  funding      2.16
  trading      0

=== 今日总盈亏（已实现含手续费 + 持仓未实现）===
本地自然日: 2026-05-07 00:00 — 现在 (begin=1778083200000 end=1778132372280 ms)

--- 1) 今日已实现 (GET /api/v5/account/bills, 无 instType=全品类, type=2) ---
账单条数: 0
Σ pnl: 0
Σ fee: 0
今日已实现(含手续费, pnl+fee): 0

--- 2) 当前未实现 (GET /api/v5/account/positions) ---
instId                                  upl       uplRatio
Σ upl (未实现合计): 0

>>> 今日总盈亏 (已实现含费 + 未实现): 0

=== 资金账户代币 (GET /api/v5/asset/balances) ===
ccy                           bal               availBal              frozenBal
USDT           2.1639051626853492     2.1639051626853492                      0
USDC            0.000000003512153      0.000000003512153                      0
FIL                  0.0000007708           0.0000007708                      0
POL           

()